In [13]:
import asyncio
from ohopilot.config import Config
from ohopilot.core.agent import Agent
from ohopilot.tools.base import ToolRegistry
from ohopilot.tools.fs import build_fs_tools

cfg = Config.load()
reg = ToolRegistry(build_fs_tools(cfg))
agent = Agent(cfg, reg)   # 暂时不传 approver（读文件不需要审批）

async def main():
    async for e in agent.run('用工具读一下 pyproject.toml 的前 3 行，然后告诉我这个项目用什么构建后端'):
        if e.type == 'text':
            print('[文字增量]', repr(e.data['delta']))
        elif e.type == 'tool_call':
            print('[调工具]', e.data['name'], e.data['arguments'])
        elif e.type == 'tool_result':
            print('[工具结果] 前80字:', e.data['content'][:80].replace(chr(10), ' | '))
        elif e.type == 'usage':
            print('[用量] total =', e.data['total_tokens'], 'tokens')
        elif e.type == 'error':
            print('[错误]', e.data['message'])
        elif e.type == 'done':
            print('[完成]')

asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop